# MNIST Unity Local Quest 3S Setup

This notebook documents how the working local Windows MNIST Unity project was adapted for **Meta Quest 3S**.

The purpose of this notebook is to reproduce the **Quest-enabled application setup**, including:

- switching the Unity project from Windows to Android;
- installing the Meta XR SDK;
- configuring Quest/OpenXR support;
- adding the Meta XR rig and passthrough-camera components;
- modifying the scene for headset execution;
- adapting the Canvas for XR use;
- adding Quest camera permission handling;
- connecting the shared MNIST model to the Quest camera pipeline;
- creating an Android/Quest build; and
- installing and launching the application through Meta Quest Developer Hub (MQDH).

Performance measurements, OVR Metrics Tool data, power observations, latency, throughput, and other Quest results are documented separately in the Quest measurement notebook.

## 1. Starting Point

The Quest implementation begins with the already working local PC Unity project.

The PC project already contains:

- the MNIST ONNX model;
- Unity Inference Engine;
- TextMeshPro;
- the benchmark scripts;
- the Canvas;
- the processed-image and frozen-image displays;
- classification controls; and
- the 28×28 MNIST preprocessing logic.

The Quest work is therefore an extension of the PC implementation rather than a separate application written from scratch.

## 2. Quest Development Environment

The Quest-enabled project used:

- **Unity Editor 6000.3.22f1**
- **Meta XR SDK All-in-One 207.0.1**
- **Unity Inference Engine 2.6.1**
- **Unity Input System 1.20.0**
- **Unity OpenXR 1.16.1**
- **Universal Render Pipeline 17.3.0**
- **Meta Quest 3S**
- **Meta Quest Developer Hub (MQDH)**

The package versions are recorded in the Quest branch `manifest.json`.

## 3. Switch the Project to Android

The PC application was originally configured for Windows.

To prepare it for Meta Quest:

1. open the Unity build interface;
2. select **Android** as the target platform;
3. switch the project to the Android platform;
4. allow Unity to reimport platform-specific assets; and
5. confirm that the Android build support modules are installed for the selected Unity Editor version.

Quest applications are distributed as Android packages, so Android must be the active build target before configuring the Quest-specific player settings.

## 4. Install the Meta XR SDK

Open:

```text
Window → Package Manager
```

Install the Meta XR SDK package used by the project.

The archived package manifest identifies:

```text
com.meta.xr.sdk.all  207.0.1
```

The Meta XR SDK provides the Quest-specific components used by this project, including:

- Meta XR camera access;
- XR rig support;
- passthrough-related functionality;
- interaction components;
- headset-specific runtime integration; and
- Quest platform support.

The Quest scripts use:

```csharp
using Meta.XR;
```

and therefore require the Meta XR SDK package to compile.

## 5. Confirm the Shared Packages

The Quest project continues to use the same core packages as the PC implementation.

Important shared dependencies include:

```text
Unity Inference Engine
TextMeshPro / Unity UI
Unity Input System
Universal Render Pipeline
```

The MNIST model is still executed locally through Unity Inference Engine.

The transition to Quest does **not** require a different neural-network model.

## 6. Enable XR and OpenXR Support

Configure the project for XR execution on Meta Quest.

The Quest-enabled package configuration includes:

```text
com.unity.xr.openxr  1.16.1
```

Enable the required Android XR/OpenXR configuration and Meta Quest support in the Unity project settings.

The goal is to ensure that:

- the Quest headset is recognized as the runtime target;
- XR rendering is enabled;
- headset tracking is available;
- controller/interaction support is available; and
- the Meta XR scene components can operate correctly.

## 7. Create the Quest Scene

The Quest-enabled scene is stored as:

```text
SecondScene.unity
```

This scene extends the original PC scene with the Meta XR objects required for headset execution.

The original PC scene remains useful as a record of the desktop implementation, while `SecondScene` represents the Quest-enabled configuration.

## 8. Add the Meta XR Rig

Add the Meta XR camera rig / XR origin structure required for Quest execution.

The rig replaces the desktop assumption that a conventional `Main Camera` alone is sufficient.

The Quest rig provides the headset-oriented camera and tracking hierarchy needed for:

- head pose;
- stereo rendering;
- XR interaction;
- tracked controllers or hands; and
- Meta-specific runtime components.

The exact hierarchy can vary with the Meta SDK version, but the important requirement is that the active Quest scene uses the Meta/XR rig rather than relying only on the original desktop camera.

## 9. Add the Quest Camera Access Component

The PC implementation uses:

```text
WebCamTexture
```

for laptop-camera input.

The Quest implementation instead uses:

```text
PassthroughCameraAccess
```

from the Meta XR SDK.

This component is assigned to the Quest camera scripts and provides access to the headset camera image.

The Quest-specific source files are:

```text
QuestCameraPreview.cs
QuestCameraMNISTInference.cs
```

## 10. Quest Camera Permission

The Quest application must request headset-camera permission before using the passthrough camera feed.

`QuestCameraPreview.cs` requests:

```text
horizonos.permission.HEADSET_CAMERA
```

The script:

1. checks whether the permission has already been granted;
2. requests it if necessary;
3. waits until the camera is active; and
4. displays the Quest camera texture in a Unity `RawImage`.

This preview stage is useful for validating camera access before running the full inference pipeline.

## 11. Replace the PC Camera Path

The PC and Quest versions differ primarily in how the source image enters Unity.

### PC

```text
Laptop Webcam
    |
    v
WebCamTexture
```

### Quest 3S

```text
Quest Passthrough Camera
    |
    v
PassthroughCameraAccess
```

After the image is available, the processing path remains intentionally similar.

## 12. Preserve the Shared MNIST Preprocessing

`QuestCameraMNISTInference.cs` preserves the same final image-processing geometry used for the PC/FPGA comparison.

The Quest source resolution is mapped to an equivalent:

```text
160 × 120
```

coordinate system.

The preprocessing uses:

```text
ROI:
x = 24 through 135
y = 4 through 115
```

and samples every four source-coordinate units.

This produces:

```text
28 × 28 = 784
```

values.

Each sampled pixel is converted to grayscale:

```text
0.299R + 0.587G + 0.114B
```

and thresholded to:

```text
0 or 1
```

before inference.

## 13. Local Quest Inference

The Quest implementation loads the same MNIST ONNX model used by the PC project.

The Quest camera script creates the worker using:

```csharp
worker = new Worker(
    runtimeModel,
    BackendType.GPUCompute
);
```

The local Quest configuration therefore performs inference on the headset rather than sending the image to the cloud.

## 14. Adapt the Canvas for Quest

The original Canvas was created for desktop interaction.

For Quest, the Canvas must be usable inside the XR scene.

The Quest scene preserves the important UI elements:

- live camera display;
- processed 28×28 display;
- frozen classification display;
- prediction/status text;
- classify button;
- frozen-image benchmark button;
- end-to-end benchmark button; and
- live-throughput button.

The Canvas components and event/input configuration must be adjusted so that the interface can be viewed and interacted with while wearing the headset.

## 15. Canvas Placement

Unlike the desktop version, the Quest interface must be positioned relative to the XR user.

A practical Quest setup places the Canvas:

- in world space or an XR-compatible presentation mode;
- in front of the user's initial viewing position;
- far enough away to be readable;
- large enough that the processed 28×28 image is visible; and
- positioned so that the buttons can be selected comfortably.

The exact transform values depend on the final scene arrangement and should be preserved by `SecondScene.unity`.

## 16. XR Interaction with the Canvas

The desktop Canvas can rely on mouse interaction.

The Quest version requires XR-compatible interaction.

The Meta/XR rig and input system should support selection of the Canvas controls through the configured Quest interaction method.

The important functional test is that the headset user can activate:

```text
Classify
Frozen Benchmark
End-to-End Benchmark
Live Throughput
```

without requiring a desktop mouse.

## 17. Connect the Quest Camera Preview

Assign the `QuestCameraPreview` component in the Quest scene.

Required references include:

```text
cameraAccess
previewImage
```

The `cameraAccess` field points to the `PassthroughCameraAccess` component.

The `previewImage` field points to the Canvas `RawImage` used to display the live Quest camera feed.

## 18. Connect the Quest Inference Script

Attach:

```text
QuestCameraMNISTInference.cs
```

to the appropriate Quest scene object.

Assign:

```text
cameraAccess
cameraDisplay
processedDisplay
frozenDisplay
predictionText
modelAsset
classifyButton
benchmarkButton
endToEndBenchmarkButton
liveThroughputButton
```

The shared `mnist_model.onnx` asset is assigned to `modelAsset`.

## 19. Validate the Quest Camera Pipeline

Before creating the final build, validate the pipeline in stages.

### Stage 1 — XR Scene

Confirm that the Quest scene loads with the correct XR rig.

### Stage 2 — UI

Confirm that the Canvas is visible and readable in the headset.

### Stage 3 — Interaction

Confirm that the Quest interaction method can activate the UI buttons.

### Stage 4 — Camera Permission

Confirm that the headset asks for camera permission when required.

### Stage 5 — Camera Preview

Confirm that the Quest camera image appears in the live preview.

### Stage 6 — Processed Image

Confirm that the 28×28 processed display updates.

### Stage 7 — Classification

Freeze the current image and confirm that a digit prediction is displayed.

## 20. Android Player Configuration

Before building, verify the Android/Quest player configuration.

Important items include:

- Android as the active platform;
- ARM64-compatible build target;
- Quest/OpenXR support enabled;
- required Meta XR components enabled;
- headset-camera permission available;
- the correct scene included in the build; and
- the project package/application identifier configured.

Use the project settings associated with the working Quest build when reproducing the application.

## 21. Include the Correct Scene

The Quest build should include:

```text
SecondScene.unity
```

as the active startup scene.

The original PC scene does not contain the complete Quest XR configuration and should not be used as the final Quest startup scene.

## 22. Build the Quest Application

Create an Android build from the Quest-enabled Unity project.

The resulting application package is an:

```text
.apk
```

The APK is not stored in this repository because the compiled application exceeds the repository upload-size limit used for this archive.

The source scene, scripts, model dependencies, and package manifest are provided so that the APK can be rebuilt.

## 23. Prepare the Quest 3S for Developer Deployment

Before installing through MQDH:

1. enable Developer Mode for the Quest device;
2. connect the Quest 3S to the development PC;
3. allow the required USB/debugging permissions in the headset;
4. confirm that the headset is visible to the development environment; and
5. confirm that Meta Quest Developer Hub recognizes the device.

The headset may prompt the user to approve USB debugging or other development permissions the first time it is connected.

## 24. Meta Quest Developer Hub

Meta Quest Developer Hub (MQDH) was used to support Quest deployment and device management.

MQDH provides a convenient desktop interface for:

- confirming that the Quest is connected;
- installing APK files;
- launching installed applications;
- viewing device information;
- collecting logs; and
- accessing additional developer tools.

The exact MQDH interface may change between versions, but the deployment workflow remains the same.

## 25. Install the APK through MQDH

After Unity produces the APK:

1. open MQDH;
2. confirm that the Quest 3S appears as a connected device;
3. use the APK installation option;
4. select the Unity-generated APK;
5. wait for installation to complete; and
6. launch the application from MQDH or from the headset's developer/unknown-sources application list.

If a newer build uses the same application identifier, MQDH can be used to install the updated APK over the development version as appropriate.

## 26. First Launch on the Quest

On the first launch:

1. allow any required runtime permissions;
2. approve headset-camera access;
3. confirm that the XR scene loads;
4. confirm that the Canvas is visible;
5. verify UI interaction;
6. verify the live camera preview;
7. verify the processed 28×28 display; and
8. run a test classification.

The application should perform local inference without requiring an Internet connection.

## 27. Quest-Specific Code Delta

Relative to the PC implementation, the major Quest-specific source additions are:

```text
QuestCameraPreview.cs
QuestCameraMNISTInference.cs
SecondScene.unity
manifest.json
```

The shared MNIST model and benchmark logic remain part of the broader Unity project.

This separation keeps the Quest code branch focused on the implementation changes required to move the working PC application onto the headset.

## 28. Reproduction Checklist

To reproduce the local Quest implementation:

```text
[ ] Open the shared MNIST Unity project
[ ] Use Unity Editor 6000.3.22f1
[ ] Switch the build target to Android
[ ] Install Meta XR SDK 207.0.1
[ ] Confirm Unity Inference Engine 2.6.1
[ ] Confirm OpenXR support
[ ] Open SecondScene
[ ] Configure the Meta XR rig
[ ] Configure PassthroughCameraAccess
[ ] Add QuestCameraPreview
[ ] Add QuestCameraMNISTInference
[ ] Configure the XR-compatible Canvas
[ ] Assign the ONNX model
[ ] Assign the camera/UI references
[ ] Verify headset-camera permission
[ ] Build the Android APK
[ ] Connect Quest 3S in developer mode
[ ] Install the APK through MQDH
[ ] Launch the application
[ ] Approve camera permission
[ ] Confirm live local classification
```

## 29. Separation from the Quest Measurement Notebook

This notebook ends once the Quest 3S application can:

- launch as a native Quest application;
- display the XR Canvas;
- access the passthrough camera;
- create the 28×28 MNIST input;
- run the ONNX model locally with GPU Compute;
- display the predicted digit; and
- be rebuilt and installed through MQDH.

The next notebook documents:

- Quest benchmark configuration;
- OVR Metrics Tool / MQDH measurement setup;
- accuracy;
- latency;
- throughput;
- memory;
- device power;
- camera benchmarks;
- screenshots and video evidence; and
- comparison with the local PC implementation.